In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import copy
import random
from typing import Type, Any, Callable, Union, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch import Tensor
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import models, transforms
from torch.hub import load_state_dict_from_url

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
train_csv_fixed = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_train_fixed.csv"
val_csv_fixed   = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_val_fixed.csv"
test_csv_fixed  = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_test_fixed.csv"

print(os.path.exists(train_csv_fixed), train_csv_fixed)
print(os.path.exists(val_csv_fixed), val_csv_fixed)
print(os.path.exists(test_csv_fixed), test_csv_fixed)

True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_train_fixed.csv
True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_val_fixed.csv
True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_test_fixed.csv


In [15]:
train_csv_fixed = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_train_fixed.csv"
val_csv_fixed   = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_val_fixed.csv"
test_csv_fixed  = "/content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_test_fixed.csv"

import os
print(os.path.exists(train_csv_fixed), train_csv_fixed)
print(os.path.exists(val_csv_fixed), val_csv_fixed)
print(os.path.exists(test_csv_fixed), test_csv_fixed)

True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_train_fixed.csv
True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_val_fixed.csv
True /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_test_fixed.csv


In [16]:
train_df = pd.read_csv(train_csv_fixed)
val_df   = pd.read_csv(val_csv_fixed)
test_df  = pd.read_csv(test_csv_fixed)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print(train_df.head())
print(train_df.columns.tolist())
print("Train label min/max:", train_df.iloc[:, 2].min(), train_df.iloc[:, 2].max())

Train: (6832, 3)
Val  : (3416, 3)
Test : (1139, 3)
   index                                          unit1_rgb  unit1_beam
0   3532  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   2224  /content/dataset/scenario23_dev/unit1/camera_d...          14
2   9416  /content/dataset/scenario23_dev/unit1/camera_d...          17
3   8510  /content/dataset/scenario23_dev/unit1/camera_d...          20
4   6877  /content/dataset/scenario23_dev/unit1/camera_d...          17
['index', 'unit1_rgb', 'unit1_beam']
Train label min/max: 2 30


In [17]:
for p in train_df.iloc[:10, 1].tolist():
    print(p, os.path.exists(str(p)))

/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2258_17_04_41.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2121_17_04_20.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_11899_18_02_04.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_8210_17_50_08.jpg False


In [18]:
class ImageBeamDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row.iloc[1]
        label = int(row.iloc[2])

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, label

In [19]:
proc_pipe = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [20]:
import pandas as pd
import os

train_df = pd.read_csv(train_csv_fixed)
print(train_df.head())

for p in train_df.iloc[:10, 1].tolist():
    print(p, os.path.exists(str(p)))

   index                                          unit1_rgb  unit1_beam
0   3532  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   2224  /content/dataset/scenario23_dev/unit1/camera_d...          14
2   9416  /content/dataset/scenario23_dev/unit1/camera_d...          17
3   8510  /content/dataset/scenario23_dev/unit1/camera_d...          20
4   6877  /content/dataset/scenario23_dev/unit1/camera_d...          17
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg False
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_225

In [21]:
sample_img, sample_label = ImageBeamDataset(train_csv_fixed, transform=proc_pipe)[0]
print("Single sample image shape:", sample_img.shape)
print("Single sample label:", sample_label)

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg'

In [22]:
import os

print("scenario23_dev exists:", os.path.exists("/content/dataset/scenario23_dev"))
print("camera_data exists:", os.path.exists("/content/dataset/scenario23_dev/unit1/camera_data"))

if os.path.exists("/content/dataset/scenario23_dev/unit1/camera_data"):
    files = os.listdir("/content/dataset/scenario23_dev/unit1/camera_data")[:10]
    print("Sample files:", files)

scenario23_dev exists: False
camera_data exists: False


bal er jibon vallage na  

In [23]:
drive_root = "/content/drive/MyDrive"

author_matches = []
for root, dirs, files in os.walk(drive_root):
    needed = {
        "scenario23_img_beam_train.csv",
        "scenario23_img_beam_val.csv",
        "scenario23_img_beam_test.csv"
    }
    if needed.issubset(set(files)):
        author_matches.append(root)

print("Author CSV folder candidates:")
for p in author_matches:
    print(p)

Author CSV folder candidates:
/content/drive/MyDrive/Image beam


In [24]:
AUTHOR_CSV_ROOT = "/content/drive/MyDrive/Image beam"

train_csv_author = os.path.join(AUTHOR_CSV_ROOT, "scenario23_img_beam_train.csv")
val_csv_author   = os.path.join(AUTHOR_CSV_ROOT, "scenario23_img_beam_val.csv")
test_csv_author  = os.path.join(AUTHOR_CSV_ROOT, "scenario23_img_beam_test.csv")

print(os.path.exists(train_csv_author), train_csv_author)
print(os.path.exists(val_csv_author), val_csv_author)
print(os.path.exists(test_csv_author), test_csv_author)

True /content/drive/MyDrive/Image beam/scenario23_img_beam_train.csv
True /content/drive/MyDrive/Image beam/scenario23_img_beam_val.csv
True /content/drive/MyDrive/Image beam/scenario23_img_beam_test.csv


In [25]:
dataset_folder_matches = []
dataset_zip_matches = []

for root, dirs, files in os.walk(drive_root):
    if "scenario23_dev" in dirs:
        dataset_folder_matches.append(os.path.join(root, "scenario23_dev"))
    for f in files:
        low = f.lower()
        if "scenario23" in low and low.endswith(".zip"):
            dataset_zip_matches.append(os.path.join(root, f))

print("Direct scenario23_dev folder matches:")
for p in dataset_folder_matches:
    print(p)

print("\nZIP matches:")
for p in dataset_zip_matches:
    print(p)

Direct scenario23_dev folder matches:

ZIP matches:
/content/drive/MyDrive/scenario23_dev_w_resources.zip


In [26]:
DATA_ROOT = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"
print("DATA_ROOT exists:", os.path.exists(DATA_ROOT), DATA_ROOT)
print("camera_data exists:", os.path.exists(os.path.join(DATA_ROOT, "unit1", "camera_data")))

DATA_ROOT exists: True /content/drive/MyDrive/scenario23_dev_w_resources.zip
camera_data exists: False


In [28]:
import zipfile

In [30]:
ZIP_PATH = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"

EXTRACT_ROOT = "/content/dataset"
os.makedirs(EXTRACT_ROOT, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_ROOT)

candidate_roots = []
for root, dirs, files in os.walk(EXTRACT_ROOT):
    if "scenario23.csv" in files:
        candidate_roots.append(root)

print("Extracted DATA_ROOT candidates:")
for c in candidate_roots:
    print(c)

DATA_ROOT = candidate_roots[0]
print("DATA_ROOT:", DATA_ROOT)
print("camera_data exists:", os.path.exists(os.path.join(DATA_ROOT, "unit1", "camera_data")))

Extracted DATA_ROOT candidates:
/content/dataset/scenario23_dev
DATA_ROOT: /content/dataset/scenario23_dev
camera_data exists: True


In [31]:
fixed_root = "/content/drive/MyDrive/scenario23_fixed_csv"
os.makedirs(fixed_root, exist_ok=True)

train_csv_fixed = os.path.join(fixed_root, "scenario23_img_beam_train_fixed.csv")
val_csv_fixed   = os.path.join(fixed_root, "scenario23_img_beam_val_fixed.csv")
test_csv_fixed  = os.path.join(fixed_root, "scenario23_img_beam_test_fixed.csv")

image_map = {}
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            image_map[f] = os.path.join(root, f)

print("Indexed images:", len(image_map))

def fix_img_csv_paths(csv_path, out_path):
    dfc = pd.read_csv(csv_path).copy()
    path_col = dfc.columns[1]
    dfc[path_col] = dfc[path_col].apply(
        lambda x: image_map.get(os.path.basename(str(x).strip()), str(x))
    )
    dfc.to_csv(out_path, index=False)
    print("Saved:", out_path)
    return dfc

fix_img_csv_paths(train_csv_author, train_csv_fixed)
fix_img_csv_paths(val_csv_author, val_csv_fixed)
fix_img_csv_paths(test_csv_author, test_csv_fixed)

Indexed images: 11387
Saved: /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_train_fixed.csv
Saved: /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_val_fixed.csv
Saved: /content/drive/MyDrive/scenario23_fixed_csv/scenario23_img_beam_test_fixed.csv


,index,unit1_rgb,unit1_beam
0,8849,/content/dataset/scenario23_dev/unit1/camera_d...,17
1,4597,/content/dataset/scenario23_dev/unit1/camera_d...,20
2,3236,/content/dataset/scenario23_dev/unit1/camera_d...,19
3,7657,/content/dataset/scenario23_dev/unit1/camera_d...,15
4,9776,/content/dataset/scenario23_dev/unit1/camera_d...,17
...,...,...,...
1134,7814,/content/dataset/scenario23_dev/unit1/camera_d...,25
1135,10956,/content/dataset/scenario23_dev/unit1/camera_d...,2
1136,906,/content/dataset/scenario23_dev/unit1/camera_d...,12
1137,5193,/content/dataset/scenario23_dev/unit1/camera_d...,9


In [32]:
train_df = pd.read_csv(train_csv_fixed)

print(train_df.head())

for p in train_df.iloc[:10, 1].tolist():
    print(p, os.path.exists(str(p)))

   index                                          unit1_rgb  unit1_beam
0   3532  /content/dataset/scenario23_dev/unit1/camera_d...          17
1   2224  /content/dataset/scenario23_dev/unit1/camera_d...          14
2   9416  /content/dataset/scenario23_dev/unit1/camera_d...          17
3   8510  /content/dataset/scenario23_dev/unit1/camera_d...          20
4   6877  /content/dataset/scenario23_dev/unit1/camera_d...          17
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3532_17_08_22.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2224_17_04_35.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10033_17_56_07.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9127_17_53_50.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_7494_17_48_19.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3825_17_09_10.jpg True
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2258_17_0

In [33]:
class ImageBeamDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row.iloc[1]
        label = int(row.iloc[2])

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, label

In [34]:
proc_pipe = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [35]:
dataset_train = ImageBeamDataset(train_csv_fixed, transform=proc_pipe)
sample_img, sample_label = dataset_train[0]

print("Single sample image shape:", sample_img.shape)
print("Single sample label:", sample_label)

Single sample image shape: torch.Size([3, 224, 224])
Single sample label: 17


In [36]:
train_loader = DataLoader(
    dataset_train,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

imgs, labels = next(iter(train_loader))
print("Batch image shape:", imgs.shape)
print("Batch label shape:", labels.shape)
print("First labels:", labels[:8])

Batch image shape: torch.Size([8, 3, 224, 224])
Batch label shape: torch.Size([8])
First labels: tensor([17, 14, 17, 20, 17, 17, 12,  9])


In [37]:
dataset_train = ImageBeamDataset(train_csv_fixed, transform=proc_pipe)
sample_img, sample_label = dataset_train[0]

print("Single sample image shape:", sample_img.shape)
print("Single sample label:", sample_label)

Single sample image shape: torch.Size([3, 224, 224])
Single sample label: 17


In [38]:
train_loader = DataLoader(
    dataset_train,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

imgs, labels = next(iter(train_loader))
print("Batch image shape:", imgs.shape)
print("Batch label shape:", labels.shape)
print("First labels:", labels[:8])

Batch image shape: torch.Size([8, 3, 224, 224])
Batch label shape: torch.Size([8])
First labels: tensor([17, 14, 17, 20, 17, 17, 12,  9])


In [ ]:
dataset_val = ImageBeamDataset(val_csv_fixed, transform=proc_pipe)
dataset_test = ImageBeamDataset(test_csv_fixed, transform=proc_pipe)

val_loader = DataLoader(
    dataset_val,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    dataset_test,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Val size:", len(dataset_val))
print("Test size:", len(dataset_test))

In [39]:
import torch
import torch.nn as nn
from torch import Tensor
from torch.hub import load_state_dict_from_url
from torchvision import models

model_urls = {
    'resnet18': 'https://download.pytorch.org/models/resnet18-5c106cde.pth',
    'resnet34': 'https://download.pytorch.org/models/resnet34-333f7ec4.pth',
    'resnet50': 'https://download.pytorch.org/models/resnet50-19c8e357.pth',
    'resnet101': 'https://download.pytorch.org/models/resnet101-5d3b4d8f.pth',
    'resnet152': 'https://download.pytorch.org/models/resnet152-b121ed2d.pth',
}

def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(
        in_planes, out_planes, kernel_size=3, stride=stride,
        padding=dilation, groups=groups, bias=False, dilation=dilation
    )

def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, dilation=1, norm_layer=None):
        super().__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        if groups != 1 or base_width != 64:
            raise ValueError("BasicBlock only supports groups=1 and base_width=64")
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 not supported in BasicBlock")

        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, dilation=1, norm_layer=None):
        super().__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        width = int(planes * (base_width / 64.0)) * groups

        self.conv1 = conv1x1(inplanes, width)
        self.bn1 = norm_layer(width)
        self.conv2 = conv3x3(width, width, stride, groups, dilation)
        self.bn2 = norm_layer(width)
        self.conv3 = conv1x1(width, planes * self.expansion)
        self.bn3 = norm_layer(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000, zero_init_residual=False,
                 groups=1, width_per_group=64, replace_stride_with_dilation=None,
                 norm_layer=None):
        super().__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self._norm_layer = norm_layer
        self.output_dim = num_classes
        self.inplanes = 64
        self.dilation = 1

        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]

        self.groups = groups
        self.base_width = width_per_group

        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = norm_layer(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, 64,  layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2,
                                       dilate=replace_stride_with_dilation[0])
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2,
                                       dilate=replace_stride_with_dilation[1])
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2,
                                       dilate=replace_stride_with_dilation[2])

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, self.output_dim)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation

        if dilate:
            self.dilation *= stride
            stride = 1

        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, self.groups,
                            self.base_width, previous_dilation, norm_layer))
        self.inplanes = planes * block.expansion

        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups,
                                base_width=self.base_width, dilation=self.dilation,
                                norm_layer=norm_layer))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        y = torch.flatten(x, 1)
        out = self.fc(y)
        return y, out

def _resnet(arch, block, layers, pretrained, progress, **kwargs):
    model = ResNet(block, layers, **kwargs)
    if pretrained:
        state_dict = load_state_dict_from_url(model_urls[arch], progress=progress)
        num_classes = kwargs.get("num_classes", 1000)

        if num_classes != 1000:
            state_dict["fc.weight"] = nn.init.xavier_normal_(model.fc.weight, gain=1)
            state_dict["fc.bias"] = nn.init.constant_(model.fc.bias, val=0)

        model.load_state_dict(state_dict)
    return model

def resnet50(pretrained=True, progress=True, **kwargs):
    return _resnet('resnet50', Bottleneck, [3, 4, 6, 3], pretrained, progress, **kwargs)

def resnet101(pretrained=True, progress=True, **kwargs):
    return _resnet('resnet101', Bottleneck, [3, 4, 23, 3], pretrained, progress, **kwargs)

def resnet152(pretrained=True, progress=True, **kwargs):
    return _resnet('resnet152', Bottleneck, [3, 8, 36, 3], pretrained, progress, **kwargs)

def build_author_resnet50():
    return resnet50(pretrained=True, progress=True, num_classes=64).to(device)

def build_author_resnet101():
    return resnet101(pretrained=True, progress=True, num_classes=64).to(device)

def build_author_resnet152():
    return resnet152(pretrained=True, progress=True, num_classes=64).to(device)

def build_vgg16():
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, 64)
    return model.to(device)

def build_vit_b16():
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    model.heads.head = nn.Linear(model.heads.head.in_features, 64)
    return model.to(device)

In [40]:
model_test = build_author_resnet50()
model_test.eval()

with torch.no_grad():
    y, out = model_test(imgs.to(device))

print("Author ResNet50 feature shape:", y.shape)
print("Author ResNet50 output shape:", out.shape)

model_vgg_test = build_vgg16()
model_vgg_test.eval()
with torch.no_grad():
    out_vgg = model_vgg_test(imgs.to(device))
print("VGG16 output shape:", out_vgg.shape)

model_vit_test = build_vit_b16()
model_vit_test.eval()
with torch.no_grad():
    out_vit = model_vit_test(imgs.to(device))
print("ViT output shape:", out_vit.shape)

Downloading: "https://download.pytorch.org/models/resnet50-19c8e357.pth" to /root/.cache/torch/hub/checkpoints/resnet50-19c8e357.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 300MB/s]


Author ResNet50 feature shape: torch.Size([8, 2048])
Author ResNet50 output shape: torch.Size([8, 64])
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 97.0MB/s] 


VGG16 output shape: torch.Size([8, 64])
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 121MB/s]  


ViT output shape: torch.Size([8, 64])


In [41]:
def unwrap_outputs(outputs):
    if isinstance(outputs, tuple):
        return outputs[1]
    return outputs

def evaluate_topk(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()
    total = 0
    correct = {k: 0 for k in ks}

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(imgs)
            outputs = unwrap_outputs(outputs)

            max_k = max(ks)
            _, pred = torch.topk(outputs, k=max_k, dim=1)
            pred = pred.t()

            total += labels.size(0)
            for k in ks:
                correct[k] += pred[:k].eq(labels.view(1, -1)).sum().item()

    return {f"top{k}": 100.0 * correct[k] / total for k in ks}

def train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=10,
    lr=1e-3,
    weight_decay=1e-4,
    milestones=(4, 8),
    save_path="/content/drive/MyDrive/best_model.pth"
):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=list(milestones), gamma=0.1)

    best_top1 = -1
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        total_train = 0

        for step, (imgs, labels) in enumerate(train_loader, start=1):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(imgs)
            outputs = unwrap_outputs(outputs)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            bs = labels.size(0)
            running_loss += loss.item() * bs
            total_train += bs

            if step % 50 == 0:
                print(f"Epoch {epoch:02d} Step {step}/{len(train_loader)} Loss {loss.item():.4f}")

        scheduler.step()

        train_loss = running_loss / total_train
        val_metrics = evaluate_topk(model, val_loader, device, ks=(1, 2, 3, 5))

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics
        })

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Top1: {val_metrics['top1']:.2f} | "
            f"Top2: {val_metrics['top2']:.2f} | "
            f"Top3: {val_metrics['top3']:.2f} | "
            f"Top5: {val_metrics['top5']:.2f}"
        )

        if val_metrics["top1"] > best_top1:
            best_top1 = val_metrics["top1"]
            torch.save(copy.deepcopy(model.state_dict()), save_path)
            print("Saved best model")

    return pd.DataFrame(history)

In [42]:
MODEL_NAME = "resnet50"   # change to: resnet50, resnet101, resnet152, vgg16, vit_b16

In [43]:
if MODEL_NAME == "resnet50":
    model = build_author_resnet50()
    lr = 1e-3
    milestones = (4, 8)
    save_path = "/content/drive/MyDrive/best_author_resnet50.pth"

elif MODEL_NAME == "resnet101":
    model = build_author_resnet101()
    lr = 1e-3
    milestones = (4, 8)
    save_path = "/content/drive/MyDrive/best_author_resnet101.pth"

elif MODEL_NAME == "resnet152":
    model = build_author_resnet152()
    lr = 1e-3
    milestones = (4, 8)
    save_path = "/content/drive/MyDrive/best_author_resnet152.pth"

elif MODEL_NAME == "vgg16":
    model = build_vgg16()
    lr = 1e-4
    milestones = (5, 8)
    save_path = "/content/drive/MyDrive/best_vgg16.pth"

elif MODEL_NAME == "vit_b16":
    model = build_vit_b16()
    lr = 1e-4
    milestones = (5, 8)
    save_path = "/content/drive/MyDrive/best_vit_b16.pth"

else:
    raise ValueError("Unknown MODEL_NAME")

print("Training model:", MODEL_NAME)

Training model: resnet50


In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=10,
    lr=lr,
    weight_decay=1e-4,
    milestones=milestones,
    save_path=save_path
)

In [ ]:
if MODEL_NAME == "resnet50":
    model_eval = build_author_resnet50()
elif MODEL_NAME == "resnet101":
    model_eval = build_author_resnet101()
elif MODEL_NAME == "resnet152":
    model_eval = build_author_resnet152()
elif MODEL_NAME == "vgg16":
    model_eval = build_vgg16()
elif MODEL_NAME == "vit_b16":
    model_eval = build_vit_b16()

model_eval.load_state_dict(torch.load(save_path, map_location=device))
test_metrics = evaluate_topk(model_eval, test_loader, device, ks=(1, 2, 3, 5))
print(f"{MODEL_NAME} Test metrics:", test_metrics)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history["epoch"], history["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Train Loss")
plt.title(f"{MODEL_NAME} Train Loss")
plt.grid(True)
plt.show()

plt.figure(figsize=(8,4))
plt.plot(history["epoch"], history["top1"], marker="o", label="Top-1")
plt.plot(history["epoch"], history["top2"], marker="o", label="Top-2")
plt.plot(history["epoch"], history["top3"], marker="o", label="Top-3")
plt.plot(history["epoch"], history["top5"], marker="o", label="Top-5")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title(f"{MODEL_NAME} Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
try:
    results
except NameError:
    results = []

results.append({
    "Model": MODEL_NAME,
    "Top-1": test_metrics["top1"],
    "Top-2": test_metrics["top2"],
    "Top-3": test_metrics["top3"],
    "Top-5": test_metrics["top5"],
})

pd.DataFrame(results)